## Video 3.2.1

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
import gradio as gr
gr.close_all()

import utils

print(f"NumPy:  {np.__version__}")
print(f"PyTorch: {torch.__version__}")

In [ ]:
device = (
    torch.accelerator.current_accelerator()
    if torch.accelerator.is_available()
    else torch.device("cpu")
)
print(f"Using: {device}")

### using Pytorch to predict a very simple linear line

In [ ]:
inputs = torch.tensor([1.0, 2.0, 3.0])
targets = torch.tensor([2.0, 4.0, 6.0])

In [ ]:
weight = torch.tensor(1.0, requires_grad=True)

In [ ]:
learning_rate = 0.1

In [ ]:
for iteration in range(1, 4):
    
    # 1. Clear the previous gradient
    weight.grad = None

    # 2. Make predictions and calculate the loss
    predictions = weight * inputs
    loss = ((predictions - targets) ** 2).mean()

    # 3. Calculate the gradient
    loss.backward()

    # 4. Update the weight without tracking the update
    old_weight = weight.item()

    with torch.no_grad():
        weight -= learning_rate * weight.grad
        new_predictions = weight * inputs
        new_loss = ((new_predictions - targets) ** 2).mean()

    # 5. Show what happened
    print(f"Iteration {iteration}")
    print(f"  W:             {old_weight:.6f} -> {weight.item():.6f}")
    print(f"  Gradient:      {weight.grad.item():.6f}")
    print(f"  Loss:   {loss.item():.6f}")
    print()

## Questions

1. How many iterations does it take the model to converge?
2. Can you make it converge faster? 
3. What happens when the learning rate is too big?

### Exercise, build a pytorch predictor to train a line to the fit the following points


In [ ]:
inputs = torch.tensor([1.0, 2.0, 3.0, 4.0])
targets = torch.tensor([8.0, 11.0, 14.0, 17.0])

## Solution

In [ ]:
weight = torch.tensor(1.0, requires_grad=True)
bias = torch.tensor(1.0, requires_grad=True)
learning_rate = 0.1

In [ ]:
for iteration in range(1, 222):
    
    # 1. Clear the previous gradient
    weight.grad = None
    bias.grad = None

    # 2. Make predictions and calculate the loss
    predictions = weight * inputs + bias
    loss = ((predictions - targets) ** 2).mean()

    # 3. Calculate the gradient
    loss.backward()

    # 4. Update the weight without tracking the update
    old_weight = weight.item()
    old_bias = bias.item()
    
    with torch.no_grad():
        weight -= learning_rate * weight.grad
        bias -= learning_rate * bias.grad
        
        new_predictions = weight * inputs + bias
        new_loss = ((new_predictions - targets) ** 2).mean()

    # 5. Show what happened
    print(f"Iteration {iteration}")
    print(f"  W:             {old_weight:.3f} -> {weight.item():.3f}")
    print(f"  B:             {old_bias:.3f} -> {bias.item():.3f}")
    print(f"  Gradient:      {weight.grad.item():.6f}")
    print(f"  Loss:   {loss.item():.6f}")
    print()

# Classifying points with neural networks

We will build a sequence of binary classifiers. Each CSV contains two input columns (`x1` and `x2`) and a `class` column containing either 0 or 1.

Our first step with a new dataset is always to plot it. When possible you always want to visualize what you can. Here we are lucky enough where everything fits nicely in one image

## Example 1: one linear decision boundary

Load and display the points before choosing a model.

In [ ]:
linear_data = pd.read_csv("data1_linear.csv")

utils.display_points(linear_data, "Dataset 1: almost linearly separable")

In [ ]:
torch.manual_seed(42)
linear_model = nn.Linear(2, 1)

In [ ]:
def train_model(model, data, epochs=500, learning_rate=0.03):
    features = torch.tensor(data[["x1", "x2"]].values, dtype=torch.float32)
    targets = torch.tensor(data["class"].values, dtype=torch.float32).reshape(-1, 1)

    loss_function = nn.BCEWithLogitsLoss()
    
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    for _ in range(epochs):
        optimizer.zero_grad()
        outputs = model(features)
        loss = loss_function(outputs, targets)
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        predictions = (torch.sigmoid(model(features)) >= 0.5).float()
        accuracy = (predictions == targets).float().mean().item()

    print(f"final loss: {loss.item():.4f}")
    print(f"accuracy:   {accuracy:.1%}")

In [ ]:
train_model(linear_model, linear_data)

utils.display_classifications(linear_model, linear_data, "A linear model fits dataset 1")

## Watching the full training, convergence process step after step

In [ ]:
from training_history import TrainingHistory

torch.manual_seed(1)  #42
linear_model = nn.Linear(2, 1)

def train_model(model, data, epochs=500, learning_rate=0.03):
    model.to(device)
    model.training_history = TrainingHistory(model)
    features = torch.tensor(data[["x1", "x2"]].values, dtype=torch.float32, device=device)
    targets = torch.tensor(data["class"].values, dtype=torch.float32, device=device).reshape(-1, 1)
    loss_function = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    for _ in range(epochs):
        optimizer.zero_grad()
        loss = loss_function(model(features), targets)
        loss.backward()
        optimizer.step()
        model.training_history.record(model)

    with torch.no_grad():
        loss = loss_function(model(features), targets)
        predictions = (torch.sigmoid(model(features)) >= 0.5).float()
        accuracy = (predictions == targets).float().mean().item()

    print(f"final loss: {loss.item():.4f}")
    print(f"accuracy:   {accuracy:.1%}")

train_model(linear_model, linear_data)


utils.display_classifications_history(linear_model, linear_data, "A linear model fits dataset 1")